# Heterogeneous Blanchard OLG — Picard-Iterated Asset Baseline

## Master's Thesis — Cecilia Trojani (UZH/ETH, 2025-2026)

---

Refinement of the structural-baseline notebook. The agent baselines
($\phi^1, \phi^2$) remain single-agent CRRA formulas (already exact in
the homogeneous limit). The asset baselines ($\phi^e, \phi^d$) now
include **one Picard iteration of the asset ODE**, which captures the
f-dynamics correction term that was missing from the single-agent
formula.

### The Picard step

The asset ODE is
$$\phi^e \cdot A_e(f) + \omega + \phi^{e\prime}(f) \mu_f(f) + \tfrac12 \phi^{e\prime\prime}(f) \sigma_f(f)^2 = 0.$$

Single-agent baseline (zeroth Picard iterate):
$$\phi^e_{(0)}(f) = \omega / (-A_e(f))$$
satisfies the **first-order** part $\phi^e A_e + \omega = 0$ but ignores
the second-order terms $\phi^{e\prime}\mu_f + \tfrac12\phi^{e\prime\prime}\sigma_f^2$.

**Picard step:** substitute $\phi^{e\prime}_{(0)}$ and $\phi^{e\prime\prime}_{(0)}$
(derivatives of the single-agent formula, computed via autograd) into
the ODE and solve for an updated $\phi^e$:
$$\phi^e_{(1)}(f) = \frac{\omega + \phi^{e\prime}_{(0)}(f) \mu_f(f) + \tfrac12 \phi^{e\prime\prime}_{(0)}(f) \sigma_f(f)^2}{-A_e(f)}.$$

This baseline captures the leading f-dynamics correction. The network
then learns a small remaining correction via the lifting envelope.

### Validation

Three sequential runs (γ¹=γ²=1, γ¹=γ²=2, γ¹=1+γ²=2) — same as the
structural baseline notebook. In the homogeneous limits both $\mu_f$
and $\sigma_f$ vanish, so the Picard correction is zero and we recover
machine-precision residuals.


In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

torch.set_default_dtype(torch.float64)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}, dtype: {torch.get_default_dtype()}')

# ── Common primitives ──────────────────────────────────────────────────────
rho   = 0.05
nu    = 0.02
mu_Y  = 0.02
sig_Y = 0.10
omega = 0.70
sig_S = sig_Y

ALPHA_POP1 = 0.5
ALPHA_POP2 = 0.5

F_EPS = 1e-3

def beta_crra(g):
    A = rho + (g-1)*mu_Y - 0.5*g*(g-1)*sig_Y**2 + nu*(1 + g + omega*(g-1))
    B = rho + (g-1)*mu_Y - 0.5*g*(g-1)*sig_Y**2 + g*nu
    disc = A**2 - 4*g*nu*omega*B
    return (A - math.sqrt(disc)) / (2*g*nu)

def homog_constants(g):
    b = beta_crra(g)
    r = rho + g*mu_Y - 0.5*g*(g+1)*sig_Y**2 + g*nu*(1-b)
    phi   = 1.0/(nu + rho/g + (g-1)/g*r + 0.5*(g-1)*sig_Y**2)
    phi_e = omega    /(r + nu - mu_Y + g*sig_Y**2)
    phi_d = (1-omega)/(r       - mu_Y + g*sig_Y**2)
    return b, r, phi, phi_e, phi_d


---
## 1. Run setup with Picard-iterated asset baselines

The key change vs the structural-baseline notebook: `phie_base(f)` and
`phid_base(f)` now apply one Picard iteration. Inside these functions
we use `torch.autograd.grad(...,  create_graph=True)` to differentiate
the single-agent formula w.r.t. f, then substitute into the Picard
correction. The full trial solution is differentiable w.r.t. f through
the nested autograd.


In [ ]:
class _MLP1D(nn.Module):
    '''Small SiLU MLP: 1 -> hidden x n_layers -> 1.'''
    def __init__(self, hidden=64, n_layers=3):
        super().__init__()
        layers = [nn.Linear(1, hidden), nn.SiLU()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden, hidden), nn.SiLU()]
        layers += [nn.Linear(hidden, 1)]
        self.net = nn.Sequential(*layers)
        for m in self.net:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)
                m.weight.data.mul_(0.1)

    def forward(self, f):
        x = f.unsqueeze(-1) if f.dim() == 1 else f
        return self.net(x).squeeze(-1)


def setup_run(gamma1, gamma2, hidden=64, n_layers=3, seed=42):
    '''Build het PINN with Picard-iterated asset baselines.'''
    torch.manual_seed(seed)
    np.random.seed(seed)

    alpha1_pref = 1.0 - 1.0/gamma1
    alpha2_pref = 1.0 - 1.0/gamma2
    b1_h = beta_crra(gamma1)
    b2_h = beta_crra(gamma2)

    _, _, PHI1_INF, PHIE_R, PHID_R = homog_constants(gamma1)
    _, _, PHI2_INF, PHIE_L, PHID_L = homog_constants(gamma2)

    # ── Equilibrium quantity helpers ──────────────────────────────────────
    def R_of_f(f):    return 1.0 / (f/gamma1 + (1-f)/gamma2)
    def P_of_f(f):
        num = f/gamma1**2 + (1-f)/gamma2**2
        den = (f/gamma1 + (1-f)/gamma2)**2
        return num/den + R_of_f(f)
    def theta_of_f(f):  return R_of_f(f) * sig_Y
    def sigma_f_of_f(f): return f * (R_of_f(f)/gamma1 - 1.0) * sig_Y
    def beta_base(f):    return (1.0 - f)*b2_h + f*b1_h
    def r_base(f):
        R = R_of_f(f); P = P_of_f(f)
        return rho + R*mu_Y - 0.5*R*P*sig_Y**2 + R*nu*(1.0 - beta_base(f))
    def mu_f_base(f):
        '''μ_f computed with heuristic β_base (for Picard baseline).'''
        R = R_of_f(f); P = P_of_f(f); beta = beta_base(f)
        beta1 = ALPHA_POP1 / (ALPHA_POP1 + ALPHA_POP2) * beta  # not really used; for symmetry
        # Just use β_base for both terms involving β¹ and β
        beta1_base = ALPHA_POP1 / (ALPHA_POP1 + ALPHA_POP2) * beta_base(f)  # split
        # Actually for simplicity use β_base for the whole μ_f formula
        Rg = R / gamma1
        term = ((Rg - 1.0)*mu_Y
                + nu * Rg * (1.0 - beta)
                + (1.0 - 0.5*Rg*P + 0.5*(1.0 + 1.0/gamma1)*R**2/gamma1 - Rg) * sig_Y**2)
        # β¹ contribution: use population-weighted share of β_base
        beta1_for_drift = ALPHA_POP1 * beta  # approximation: half of total β goes to type 1
        return nu * (alpha1_pref * beta1_for_drift - f) + f * term

    # ── Agent baselines (single-agent CRRA at r_base) ────────────────────
    def phi1_base(f):
        r = r_base(f)
        return 1.0/(nu + rho/gamma1 + (gamma1-1.0)/gamma1*r + 0.5*(gamma1-1.0)*sig_Y**2)
    def phi2_base(f):
        r = r_base(f)
        return 1.0/(nu + rho/gamma2 + (gamma2-1.0)/gamma2*r + 0.5*(gamma2-1.0)*sig_Y**2)

    # ── Asset Picard baselines ────────────────────────────────────────────
    def _phie_zeroth(f):
        '''Single-agent zeroth-iterate asset baseline.'''
        r = r_base(f); R = R_of_f(f)
        return omega / (r + nu - mu_Y + R*sig_Y**2)
    def _phid_zeroth(f):
        r = r_base(f); R = R_of_f(f)
        return (1.0 - omega) / (r - mu_Y + R*sig_Y**2)

    def phie_base(f):
        '''Picard-iterated asset baseline for equity. Uses inner autograd.'''
        # Make f a leaf to differentiate against
        if not f.requires_grad:
            f_inner = f.detach().requires_grad_(True)
        else:
            f_inner = f

        phi_e_0 = _phie_zeroth(f_inner)
        g_one = torch.ones_like(phi_e_0)
        dphi_e_0, = torch.autograd.grad(phi_e_0, f_inner, g_one, create_graph=True)
        d2phi_e_0, = torch.autograd.grad(dphi_e_0, f_inner, g_one, create_graph=True)

        # Picard correction with β_base dynamics
        muf = mu_f_base(f_inner)
        sf  = sigma_f_of_f(f_inner)
        r   = r_base(f_inner); R = R_of_f(f_inner)
        A_e = -nu + mu_Y - r - R*sig_Y * sig_Y  # = -ν+μ_Y-r-θσ_Y, θ=Rσ_Y so θσ_Y = R σ²

        return (omega + dphi_e_0 * muf + 0.5 * d2phi_e_0 * sf**2) / (-A_e)

    def phid_base(f):
        '''Picard-iterated asset baseline for dividend.'''
        if not f.requires_grad:
            f_inner = f.detach().requires_grad_(True)
        else:
            f_inner = f

        phi_d_0 = _phid_zeroth(f_inner)
        g_one = torch.ones_like(phi_d_0)
        dphi_d_0, = torch.autograd.grad(phi_d_0, f_inner, g_one, create_graph=True)
        d2phi_d_0, = torch.autograd.grad(dphi_d_0, f_inner, g_one, create_graph=True)

        muf = mu_f_base(f_inner)
        sf  = sigma_f_of_f(f_inner)
        r   = r_base(f_inner); R = R_of_f(f_inner)
        # For dividend: A_d = μ_Y - r - θ σ_Y (no -ν)
        A_d = mu_Y - r - R*sig_Y * sig_Y

        return ((1.0 - omega) + dphi_d_0 * muf + 0.5 * d2phi_d_0 * sf**2) / (-A_d)

    # ── Boundary anchor correction ───────────────────────────────────────
    # The Picard baseline may not exactly equal homog anchor at boundaries
    # because μ_f(1) ≠ 0 in general. We add a linear correction to ensure
    # baselines match anchors exactly at f=0 and f=1.
    with torch.enable_grad():
        f0 = torch.tensor([0.0])
        f1 = torch.tensor([1.0])
        phie_picard_at_0 = phie_base(f0).item()
        phie_picard_at_1 = phie_base(f1).item()
        phid_picard_at_0 = phid_base(f0).item()
        phid_picard_at_1 = phid_base(f1).item()
    print(f'  Picard baseline values:')
    print(f'    φᵉ_picard(0) = {phie_picard_at_0:.4f}    anchor {PHIE_L:.4f}    Δ={phie_picard_at_0-PHIE_L:+.4f}')
    print(f'    φᵉ_picard(1) = {phie_picard_at_1:.4f}    anchor {PHIE_R:.4f}    Δ={phie_picard_at_1-PHIE_R:+.4f}')
    print(f'    φᵈ_picard(0) = {phid_picard_at_0:.4f}    anchor {PHID_L:.4f}    Δ={phid_picard_at_0-PHID_L:+.4f}')
    print(f'    φᵈ_picard(1) = {phid_picard_at_1:.4f}    anchor {PHID_R:.4f}    Δ={phid_picard_at_1-PHID_R:+.4f}')

    phie_corr_L = PHIE_L - phie_picard_at_0
    phie_corr_R = PHIE_R - phie_picard_at_1
    phid_corr_L = PHID_L - phid_picard_at_0
    phid_corr_R = PHID_R - phid_picard_at_1

    def phie_base_corrected(f):
        return phie_base(f) + (1.0 - f)*phie_corr_L + f*phie_corr_R
    def phid_base_corrected(f):
        return phid_base(f) + (1.0 - f)*phid_corr_L + f*phid_corr_R

    # ── Network ──────────────────────────────────────────────────────────
    class HardBC_Het_Picard(nn.Module):
        def __init__(self):
            super().__init__()
            self.N1 = _MLP1D(hidden, n_layers)
            self.N2 = _MLP1D(hidden, n_layers)
            self.Ne = _MLP1D(hidden, n_layers)
            self.Nd = _MLP1D(hidden, n_layers)

        def forward(self, f):
            n1 = self.N1(f); n2 = self.N2(f); ne = self.Ne(f); nd = self.Nd(f)
            phi1 = phi1_base(f) * torch.exp((1.0 - f) * n1)
            phi2 = phi2_base(f) * torch.exp(f * n2)
            phie = phie_base_corrected(f) * torch.exp(f * (1.0 - f) * ne)
            phid = phid_base_corrected(f) * torch.exp(f * (1.0 - f) * nd)
            return phi1, phi2, phie, phid

    net = HardBC_Het_Picard().to(device)

    # ── Residual function (endogenous β) ──────────────────────────────────
    def compute_residuals(net, f):
        phi1, phi2, phie, phid = net(f)
        g_one = torch.ones_like(phi1)
        dphi1, = torch.autograd.grad(phi1, f, grad_outputs=g_one, create_graph=True)
        dphi2, = torch.autograd.grad(phi2, f, grad_outputs=g_one, create_graph=True)
        dphie, = torch.autograd.grad(phie, f, grad_outputs=g_one, create_graph=True)
        dphid, = torch.autograd.grad(phid, f, grad_outputs=g_one, create_graph=True)
        d2phi1, = torch.autograd.grad(dphi1, f, grad_outputs=g_one, create_graph=True)
        d2phi2, = torch.autograd.grad(dphi2, f, grad_outputs=g_one, create_graph=True)
        d2phie, = torch.autograd.grad(dphie, f, grad_outputs=g_one, create_graph=True)
        d2phid, = torch.autograd.grad(dphid, f, grad_outputs=g_one, create_graph=True)

        b1 = ALPHA_POP1 * phie / phi1
        b2 = ALPHA_POP2 * phie / phi2
        b = b1 + b2

        R = R_of_f(f); P = P_of_f(f); th = theta_of_f(f)
        r = rho + R*mu_Y - 0.5*R*P*sig_Y**2 + R*nu*(1.0 - b)
        sf = sigma_f_of_f(f); half_sf2 = 0.5 * sf**2

        Rg = R / gamma1
        theta_bracket = ((Rg - 1.0)*mu_Y
                       + nu * Rg * (1.0 - b)
                       + (1.0 - 0.5*Rg*P + 0.5*(1.0 + 1.0/gamma1)*R**2/gamma1 - Rg) * sig_Y**2)
        # μ_f formula (Prop 3.3 / eq. 5.12): ν(β¹ − f) + f·[bracket]
        # Note: β¹ already includes the population share α¹_pop via b1 definition.
        muf = nu * (b1 - f) + f * theta_bracket

        def A_of(g):
            return (-nu - rho/g
                    + (1.0 - 1.0/g)*(-r)
                    + 0.5*(1.0 - 1.0/g)*(-1.0/g)*th**2)

        L1 = phi1 * A_of(gamma1) + 1.0 + dphi1 * muf + d2phi1 * half_sf2
        L2 = phi2 * A_of(gamma2) + 1.0 + dphi2 * muf + d2phi2 * half_sf2
        Le = phie * (-nu + mu_Y - r - th*sig_Y) + omega       + dphie * muf + d2phie * half_sf2
        Ld = phid * (       mu_Y - r - th*sig_Y) + (1.0-omega) + dphid * muf + d2phid * half_sf2
        Lmc = f*phi1 + (1.0 - f)*phi2 - phie - phid
        return L1, L2, Le, Ld, Lmc

    constants = {
        'gamma1': gamma1, 'gamma2': gamma2,
        'PHI1_INF': PHI1_INF, 'PHI2_INF': PHI2_INF,
        'PHIE_L': PHIE_L, 'PHIE_R': PHIE_R,
        'PHID_L': PHID_L, 'PHID_R': PHID_R,
        'b1_homog': b1_h, 'b2_homog': b2_h,
    }
    return net, compute_residuals, constants

print('setup_run with Picard baselines defined.')


---
## 2. Training & validation (same as structural baseline notebook)


In [ ]:
def sample_f(n):
    n_bdy_l = int(0.20 * n); n_bdy_r = int(0.20 * n)
    n_int   = n - n_bdy_l - n_bdy_r
    f_bdy_l = F_EPS + (0.10 - F_EPS) * torch.rand(n_bdy_l, device=device)
    f_bdy_r = 0.90  + (1.0 - F_EPS - 0.90) * torch.rand(n_bdy_r, device=device)
    f_int   = F_EPS + (1.0 - 2*F_EPS) * torch.rand(n_int, device=device)
    return torch.cat([f_bdy_l, f_int, f_bdy_r]).requires_grad_(True)


def train_run(net, compute_residuals, n_adam=2000, n_lbfgs=50, n_coll=4000,
              lr=1e-4, w_agent=1.0, w_asset=1.0, w_mc=0.5, verbose_every=200):
    opt = optim.Adam(net.parameters(), lr=lr)
    sched = optim.lr_scheduler.MultiStepLR(opt, milestones=[int(0.5*n_adam), int(0.83*n_adam)], gamma=0.4)
    print(f'  Adam: {n_adam} ep, lr={lr}, N_coll={n_coll}, w(agent,asset,mc)=({w_agent},{w_asset},{w_mc})')
    for ep in range(1, n_adam + 1):
        f = sample_f(n_coll)
        L1, L2, Le, Ld, Lmc = compute_residuals(net, f)
        loss = (w_agent*(L1.pow(2).mean() + L2.pow(2).mean())
              + w_asset*(Le.pow(2).mean() + Ld.pow(2).mean())
              + w_mc   * Lmc.pow(2).mean())
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=10.0)
        opt.step(); sched.step()
        if ep % verbose_every == 0 or ep == 1:
            print(f'    ep {ep:5d}  loss={loss.item():.4e}  '
                  f'L1²={L1.pow(2).mean().item():.2e}  L2²={L2.pow(2).mean().item():.2e}  '
                  f'Le²={Le.pow(2).mean().item():.2e}  Ld²={Ld.pow(2).mean().item():.2e}  '
                  f'mc²={Lmc.pow(2).mean().item():.2e}')

    print(f'  L-BFGS: max {n_lbfgs} outer iters, residual-weighted grid')
    N_CAND = 20000; N_FROZEN = 8000
    f_cand = sample_f(N_CAND).detach()
    f_cand_g = f_cand.clone().requires_grad_(True)
    L1c, L2c, Lec, Ldc, Lmcc = compute_residuals(net, f_cand_g)
    with torch.no_grad():
        res_total = (w_agent*(L1c.pow(2)+L2c.pow(2)) + w_asset*(Lec.pow(2)+Ldc.pow(2)) + w_mc*Lmcc.pow(2))
        weights = res_total.detach().abs().clamp(min=1e-16)
        weights = weights / weights.sum()
        idx = torch.multinomial(weights, N_FROZEN, replacement=False)
        f_grid = f_cand[idx].detach().requires_grad_(True)

    opt2 = optim.LBFGS(net.parameters(), lr=1.0, max_iter=20,
                       tolerance_grad=1e-14, tolerance_change=1e-16,
                       history_size=50, line_search_fn='strong_wolfe')
    prev_loss = float('inf')
    for outer in range(n_lbfgs):
        def closure():
            opt2.zero_grad()
            L1, L2, Le, Ld, Lmc = compute_residuals(net, f_grid)
            loss = (w_agent*(L1.pow(2).mean() + L2.pow(2).mean())
                  + w_asset*(Le.pow(2).mean() + Ld.pow(2).mean())
                  + w_mc   * Lmc.pow(2).mean())
            loss.backward(); return loss
        loss = opt2.step(closure).item()
        if outer % 10 == 0 or outer < 5:
            print(f'    outer {outer:3d}   loss={loss:.6e}')
        if abs(prev_loss - loss) < 1e-13:
            print(f'    stagnated at outer {outer}'); break
        prev_loss = loss

    print(f'  Final L-BFGS loss: {loss:.6e}')
    return loss


def validate_run(net, compute_residuals, constants, n_plot=500, title_suffix='', save_prefix=None):
    net.eval()
    g1, g2 = constants['gamma1'], constants['gamma2']
    with torch.no_grad():
        f_bc = torch.tensor([0.0, 1.0], device=device)
        p1, p2, pe, pd = net(f_bc)
    print(f'\n  === Boundary check (γ¹={g1}, γ²={g2}) ===')
    print(f'    φ¹(1) = {p1[1].item():.6f}    target = {constants["PHI1_INF"]:.6f}    err = {abs(p1[1].item() - constants["PHI1_INF"]):.1e}')
    print(f'    φ²(0) = {p2[0].item():.6f}    target = {constants["PHI2_INF"]:.6f}    err = {abs(p2[0].item() - constants["PHI2_INF"]):.1e}')
    print(f'    φᵉ(0) = {pe[0].item():.6f}    target = {constants["PHIE_L"]:.6f}    err = {abs(pe[0].item() - constants["PHIE_L"]):.1e}')
    print(f'    φᵉ(1) = {pe[1].item():.6f}    target = {constants["PHIE_R"]:.6f}    err = {abs(pe[1].item() - constants["PHIE_R"]):.1e}')
    print(f'    φᵈ(0) = {pd[0].item():.6f}    target = {constants["PHID_L"]:.6f}    err = {abs(pd[0].item() - constants["PHID_L"]):.1e}')
    print(f'    φᵈ(1) = {pd[1].item():.6f}    target = {constants["PHID_R"]:.6f}    err = {abs(pd[1].item() - constants["PHID_R"]):.1e}')
    print(f'    φ¹(0) = {p1[0].item():.6f}    (off-side, learned)')
    print(f'    φ²(1) = {p2[1].item():.6f}    (off-side, learned)')

    f_plot = (F_EPS + (1-2*F_EPS) * torch.linspace(0, 1, n_plot, device=device)).requires_grad_(True)
    L1, L2, Le, Ld, Lmc = compute_residuals(net, f_plot)
    print(f'  === Residual summary (interior, N={n_plot}) ===')
    for name, L in [('L_φ¹', L1), ('L_φ²', L2), ('L_φᵉ', Le), ('L_φᵈ', Ld), ('L_mc', Lmc)]:
        arr = L.abs().detach().cpu().numpy()
        print(f'    {name}:  mean={arr.mean():.4e}    max={arr.max():.4e}')

    if abs(g1 - g2) < 1e-12:
        phi_homog = constants['PHI1_INF']
        with torch.no_grad():
            p1_a, p2_a, pe_a, pd_a = net(f_plot.detach())
        print(f'  === Homog-limit check ===')
        print(f'    max |φ¹(f) - {phi_homog:.4f}| = {(p1_a - phi_homog).abs().max().item():.4e}')
        print(f'    max |φ²(f) - {phi_homog:.4f}| = {(p2_a - phi_homog).abs().max().item():.4e}')

    # Plot
    f_np = f_plot.detach().cpu().numpy()
    with torch.no_grad():
        phi1, phi2, phie, phid = net(f_plot.detach())
    phi1_np = phi1.cpu().numpy(); phi2_np = phi2.cpu().numpy()
    phie_np = phie.cpu().numpy(); phid_np = phid.cpu().numpy()
    mc_np = f_np*phi1_np + (1-f_np)*phi2_np - phie_np - phid_np

    # Absolute residuals
    L1_abs = L1.abs().detach().cpu().numpy()
    L2_abs = L2.abs().detach().cpu().numpy()
    Le_abs = Le.abs().detach().cpu().numpy()
    Ld_abs = Ld.abs().detach().cpu().numpy()

    # Relative residuals: |L_phi^i| / |phi^i| (dimensionless equation imbalance)
    eps = 1e-16
    L1_rel = L1_abs / (np.abs(phi1_np) + eps)
    L2_rel = L2_abs / (np.abs(phi2_np) + eps)
    Le_rel = Le_abs / (np.abs(phie_np) + eps)
    Ld_rel = Ld_abs / (np.abs(phid_np) + eps)
    mc_scale = np.abs(f_np*phi1_np + (1-f_np)*phi2_np) + np.abs(phie_np + phid_np)
    mc_rel = np.abs(mc_np) / (0.5 * mc_scale + eps)

    print(f'  === Relative residual summary (|L_φ^i|/|φ^i|, interior) ===')
    for name, R in [('L_φ¹/φ¹', L1_rel), ('L_φ²/φ²', L2_rel),
                    ('L_φᵉ/φᵉ', Le_rel), ('L_φᵈ/φᵈ', Ld_rel),
                    ('L_mc/scale', mc_rel)]:
        print(f'    {name}:  mean={R.mean():.4e}    max={R.max():.4e}')

    fig, axes = plt.subplots(2, 3, figsize=(16, 8))
    ax = axes[0,0]
    ax.plot(f_np, phi1_np, lw=2, color='steelblue', label=r'$\widehat\phi^1$')
    ax.plot(f_np, phi2_np, lw=2, color='firebrick', label=r'$\widehat\phi^2$')
    ax.scatter([1.0], [constants['PHI1_INF']], s=70, marker='o',
               facecolor='none', edgecolor='steelblue', lw=1.8, zorder=5,
               label=r'anchor $\phi^1_\infty$')
    ax.scatter([0.0], [constants['PHI2_INF']], s=70, marker='o',
               facecolor='none', edgecolor='firebrick', lw=1.8, zorder=5,
               label=r'anchor $\phi^2_\infty$')
    ax.set_xlabel('f'); ax.set_ylabel(r'$\phi^i(f)$'); ax.set_title('Agent WCR')
    ax.ticklabel_format(useOffset=False, style='plain', axis='y')
    ax.legend(fontsize=8, loc='best'); ax.grid(alpha=0.3)

    ax = axes[0,1]
    ax.plot(f_np, phie_np, lw=2, color='seagreen', label=r'$\widehat\phi^e$')
    ax.plot(f_np, phid_np, lw=2, color='darkorange', label=r'$\widehat\phi^d$')
    ax.plot(f_np, phie_np + phid_np, lw=1.5, ls='--', color='black', label=r'$\phi^e+\phi^d$')
    ax.scatter([0.0, 1.0], [constants['PHIE_L'], constants['PHIE_R']], s=60, marker='o',
               facecolor='none', edgecolor='seagreen', lw=1.8, zorder=5)
    ax.scatter([0.0, 1.0], [constants['PHID_L'], constants['PHID_R']], s=60, marker='o',
               facecolor='none', edgecolor='darkorange', lw=1.8, zorder=5)
    ax.set_xlabel('f'); ax.set_ylabel(r'$\phi(f)$'); ax.set_title('Asset prices')
    ax.ticklabel_format(useOffset=False, style='plain', axis='y')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

    ax = axes[0,2]
    ax.plot(f_np, mc_np, lw=2, color='purple')
    ax.axhline(0, color='gray', ls=':', alpha=0.6)
    ax.set_xlabel('f'); ax.set_ylabel('mc residual'); ax.set_title('Market clearing (absolute)')
    ax.grid(alpha=0.3)

    ax = axes[1,0]
    for name, L_abs, col in [(r'$L_{\phi^1}$', L1_abs, 'steelblue'), (r'$L_{\phi^2}$', L2_abs, 'firebrick'),
                             (r'$L_{\phi^e}$', Le_abs, 'seagreen'),  (r'$L_{\phi^d}$', Ld_abs, 'darkorange')]:
        ax.semilogy(f_np, L_abs + 1e-16, lw=1.2, label=name, color=col)
    ax.set_xlabel('f'); ax.set_ylabel('|L|'); ax.set_title('ODE residuals — absolute (log)')
    ax.legend(fontsize=9); ax.grid(alpha=0.3, which='both')

    ax = axes[1,1]
    for name, R, col in [(r'$|L_{\phi^1}|/\phi^1$', L1_rel, 'steelblue'),
                         (r'$|L_{\phi^2}|/\phi^2$', L2_rel, 'firebrick'),
                         (r'$|L_{\phi^e}|/\phi^e$', Le_rel, 'seagreen'),
                         (r'$|L_{\phi^d}|/\phi^d$', Ld_rel, 'darkorange')]:
        ax.semilogy(f_np, R + 1e-16, lw=1.2, label=name, color=col)
    ax.set_xlabel('f'); ax.set_ylabel('|L|/|φ|'); ax.set_title('ODE residuals — relative (log)')
    ax.legend(fontsize=9); ax.grid(alpha=0.3, which='both')

    ax = axes[1,2]
    ax.semilogy(f_np, mc_rel + 1e-16, lw=1.5, color='purple')
    ax.set_xlabel('f'); ax.set_ylabel(r'$|L_{mc}|/\frac{1}{2}(|\sum f\phi^i|+|\phi^e+\phi^d|)$')
    ax.set_title('Market clearing — relative (log)')
    ax.grid(alpha=0.3, which='both')

    plt.suptitle(rf'Picard baseline ($\gamma^1={g1}$, $\gamma^2={g2}$){title_suffix}',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    if save_prefix:
        plt.savefig(f'{save_prefix}.png', dpi=140, bbox_inches='tight')
    plt.show()

print('train_run and validate_run defined.')


---
## 3. Run 1 — Homog log (γ¹=γ²=1)

In the homog limit, $\mu_f \equiv 0$ and $\sigma_f \equiv 0$, so the
Picard correction is zero and we recover the single-agent baseline.
Expect machine-precision residuals.


In [ ]:
print('=' * 70)
print('RUN 1: γ¹ = γ² = 1 (homog log) — Picard baseline')
print('=' * 70)

net1, cr1, c1 = setup_run(gamma1=1.0, gamma2=1.0)
final1 = train_run(net1, cr1, n_adam=2000, n_lbfgs=50)
validate_run(net1, cr1, c1, title_suffix=' — RUN 1', save_prefix='run1_picard_log')


---
## 4. Run 2 — Homog CRRA (γ¹=γ²=2)


In [ ]:
print('=' * 70)
print('RUN 2: γ¹ = γ² = 2 (homog CRRA) — Picard baseline')
print('=' * 70)

net2, cr2, c2 = setup_run(gamma1=2.0, gamma2=2.0)
final2 = train_run(net2, cr2, n_adam=2000, n_lbfgs=50)
validate_run(net2, cr2, c2, title_suffix=' — RUN 2', save_prefix='run2_picard_crra')


---
## 5. Run 3 — True heterogeneous (γ¹=1, γ²=2) — PRODUCTION

This is the case where the Picard correction should make a real
difference, because $\mu_f, \sigma_f \neq 0$ in the interior. The
single-agent baseline misses the f-dynamics correction; the Picard
baseline captures it analytically.


In [ ]:
print('=' * 70)
print('RUN 3: γ¹ = 1, γ² = 2 (true heterogeneous) — Picard baseline')
print('=' * 70)

net3, cr3, c3 = setup_run(gamma1=1.0, gamma2=2.0)
final3 = train_run(net3, cr3, n_adam=3000, n_lbfgs=100)
validate_run(net3, cr3, c3, title_suffix=' — RUN 3 (production)', save_prefix='run3_picard_het')


---
## 6. Summary


In [ ]:
print('=' * 70)
print('SUMMARY OF ALL RUNS — Picard Baseline')
print('=' * 70)
print(f'  Run 1 (γ¹=γ²=1, homog log):  final loss = {final1:.6e}')
print(f'  Run 2 (γ¹=γ²=2, homog CRRA): final loss = {final2:.6e}')
print(f'  Run 3 (γ¹=1, γ²=2, het):     final loss = {final3:.6e}')

torch.save(net3.state_dict(), 'net_het_picard_g1_g2.pt')
print(f'\nSaved production network -> net_het_picard_g1_g2.pt')
